In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from pprint import pprint
from pydrake.all import (
    ConstantVectorSource,
    DiagramBuilder,
    QuasidynamicPlanarPlant,
    RigidTransform,
    RigidTransform2d,
    RollPitchYaw,
    Simulator,
    Rgba,
    Meshcat,
    Cylinder,
    VectorLogSink,
    Rectangle,
)
from algorithms import (
    IterativeSLSController,
    TrajOptProblem,
)
from models.iiwa import AddPlanarIiwa
IS_TEST = "BAZEL_TEST" in os.environ

In [ ]:
meshcat = Meshcat()

In [ ]:
dt = 0.02
plant = QuasidynamicPlanarPlant(dt=dt)

mu = 0.5
joint_lb1, joint_ub1 = AddPlanarIiwa(
    plant,
    name = "left_arm",
    X_WR=RigidTransform2d(np.pi/2, [-0.65, 0]),
    friction_coefficient=mu,
    actuation_stiffness=[10, 5, 5],
)
joint_lb2, joint_ub2 = AddPlanarIiwa(
    plant,
    name = "right_arm",
    X_WR=RigidTransform2d(np.pi/2, [0.65, 0]),
    friction_coefficient=mu,
    actuation_stiffness=[10, 5, 5],
)
joint_lb = np.concatenate((joint_lb1, joint_lb2))
joint_ub = np.concatenate((joint_ub1, joint_ub2))

object = plant.AddRigidBody(mass=0.05, p_BBcm_B=[0,0], I_BBcm=0.008)
plant.RegisterCollisionGeometry(object, RigidTransform2d(), Rectangle(0.25, 0.25), mu)
plant.RegisterVisualGeometry(object, RigidTransform2d(), Rectangle(0.25, 0.25), Rgba(0,0.6,1,1), "bucket/bucket")

axis = Cylinder(0.002, 0.06)
plant.RegisterVisualGeometry(object, RigidTransform(RollPitchYaw(0, np.pi/2, 0), [0.03, 0, 0.05]), axis, Rgba(1,0,0,1), "bucket/x")
plant.RegisterVisualGeometry(object, RigidTransform(RollPitchYaw(np.pi/2, 0, 0), [0, 0.03, 0.05]), axis, Rgba(0,1,0,1), "bucket/y")
plant.RegisterVisualGeometry(object, RigidTransform(RollPitchYaw(0, 0, np.pi/2), [0,    0, 0.08]), axis, Rgba(0,0,1,1), "bucket/z")

plant.Finalize()
plant.SetMeshcat(meshcat, z_height=0.1)

In [ ]:
u0 = np.deg2rad([10, -80, -20, -10, 80, 20])
x0 = np.concatenate((u0, [0.0, 0.52, np.deg2rad(0)]))
x_target = np.concatenate((np.zeros(6), [0.0, 0.3, np.deg2rad(90)]))

builder = DiagramBuilder()
plant = plant.Clone()
builder.AddSystem(plant)
source = ConstantVectorSource(u0)
builder.AddSystem(source)
builder.Connect(source.get_output_port(), plant.get_actuation_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)

plant_context = diagram.GetMutableSubsystemContext(
    plant, simulator.get_context()
)
plant.SetState(plant_context, x0)

simulator.AdvanceTo(0)

In [ ]:
context = plant.CreateDefaultContext()
plant.set_dynamics_smoothing(context, 1e-3)
plant.set_geometry_smoothing(context, 1e-3)

num_steps = 100
prob = TrajOptProblem(system=plant, context=context, num_steps=num_steps)
prob.SetInitialState(x0)

x = prob.state()
u = prob.input()
Q = np.diag([0, 0, 0, 0, 0, 0, 1, 1, 1])
R = np.identity(6) * 0.1
prob.AddStageCost((x - x_target).dot(Q @ (x - x_target)) + u.dot(R @ u))
prob.AddTerminalCost(1e3 * (x - x_target).dot(Q @ (x - x_target)))

for k in range(num_steps - 1):
    diff = prob.input(k) - prob.input(k + 1)
    prob.AddCost(100 * diff.dot(diff))

prob.AddConstraint(prob.input(0), lb=u0, ub=u0)

joint_ub = np.deg2rad([170, 120, 120] * 2)
joint_lb = -joint_ub
G = np.hstack((np.zeros((len(u), len(x))), np.identity(len(u))))
prob.AddStageLinearConstraint(G=G, g=-joint_ub)
prob.AddStageLinearConstraint(G=-G, g=joint_lb)

prob.SetLinearCollisionConstraint(enable=True, distance_threshold=0.0)

Q_bar = np.identity(9) * 1e1
R_bar = np.identity(6) * 1e1
Qf_bar = np.identity(9) * 1e2
prob.SetUncertaintyTubeCost(Q_bar=Q_bar, R_bar=R_bar, Qf_bar=Qf_bar)

In [ ]:
us_init = np.repeat(u0[np.newaxis, :], repeats=num_steps, axis=0)
controller = IterativeSLSController(
    prob,
    us_init=us_init,
    delta_cap=0.03,
    max_outer_iters=30,
)

In [ ]:
builder = DiagramBuilder()
plant = builder.AddSystem(plant.Clone())
builder.AddSystem(controller)
logger = builder.AddSystem(VectorLogSink(9))
builder.Connect(plant.get_state_output_port(), controller.get_input_port())
builder.Connect(controller.get_output_port(), plant.get_actuation_input_port())
builder.Connect(plant.get_state_output_port(), logger.get_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)

plant_context = diagram.GetMutableSubsystemContext(
    plant, simulator.get_context()
)
plant.SetState(plant_context, x0)
plant.set_dynamics_smoothing(plant_context, 0.0)
plant.set_geometry_smoothing(plant_context, 0.0)

meshcat.StartRecording()
simulator.AdvanceTo(num_steps * dt)
meshcat.StopRecording()
meshcat.PublishRecording()

plt.figure(figsize=(15, 4), constrained_layout=True)
plt.suptitle("IterativeSLS")
for i in range(3):
    plt.subplot(1, 3, i + 1)
    dof = i + 6

    ts = np.arange(num_steps + 1) * dt
    plt.plot(ts, controller.xs[:, dof], label="Nominal trajectory")
    plt.fill_between(
        ts, controller.xs_lb[:, dof], controller.xs_ub[:, dof], alpha=0.3
    )

    log = logger.FindLog(simulator.get_context())
    plt.plot(log.sample_times(), log.data()[dof], label="Feedback control")

    plt.axhline(y=x_target[dof], color="k", linestyle="--", linewidth=1.0)

    plt.xlabel("$t$")
    plt.ylabel("${}_o$".format(["x", "y", r"\theta"][i]))
    plt.legend()
    plt.xlim([0, num_steps * dt])
plt.show()